# 04b — Create UC Analytics Functions

**Jackson and Jackson HR Digital**

Registers parameterised SQL table-valued functions in Unity Catalog so the Hire Right agent
can answer HR analytics questions without going through Genie.
Each function wraps a targeted query against the Gold `candidate_scoring_summary` table.

### Functions created
| Function | Purpose |
|---|---|
| `get_candidate(candidate_id)` | Full profile + all scores for a single candidate |
| `get_top_candidates(job_id, top_n)` | Top N candidates for a role ranked by total_score |
| `get_candidates_by_job(job_id)` | All candidates assigned to a given job |
| `get_pipeline_candidates()` | Active pipeline only — candidates with `hired IS NULL` (C011–C020) |
| `get_hire_analytics()` | Aggregate hire rate, avg scores, and candidate counts by job |

### Prerequisites
- Notebooks `00` – `03` have been run (`candidate_scoring_summary` Gold table exists)

**Next notebook:** `04_create_genie_space.ipynb`

## Setup

In [ ]:
import os

dbutils.widgets.text("catalog", "bx4",      "UC Catalog")
dbutils.widgets.text("schema",  "hrd_2030", "UC Schema")

catalog = dbutils.widgets.get("catalog") or os.getenv("TARGET_CATALOG", "bx4")
schema  = dbutils.widgets.get("schema")  or os.getenv("TARGET_SCHEMA",  "hrd_2030")

print(f"catalog : {catalog}")
print(f"schema  : {schema}")

## Create UC SQL Functions

In [ ]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.get_candidate(candidate_id STRING)
RETURNS TABLE (
  candidate_id              STRING,
  first_name                STRING,
  last_name                 STRING,
  email                     STRING,
  current_title             STRING,
  current_company           STRING,
  years_total_experience    INT,
  years_hr_experience       INT,
  years_leadership          INT,
  education_level           STRING,
  certifications            STRING,
  job_id                    STRING,
  job_title                 STRING,
  education_score           INT,
  experience_score          INT,
  leadership_score          INT,
  certification_score       INT,
  skills_match_score        INT,
  industry_relevance_score  INT,
  interview_score           INT,
  culture_fit               INT,
  total_score               DOUBLE,
  hired                     INT
)
COMMENT 'Returns full profile and all feature scores for a single candidate. Pass candidate_id e.g. C001.'
RETURN
  SELECT
    candidate_id, first_name, last_name, email,
    current_title, current_company,
    years_total_experience, years_hr_experience, years_leadership,
    education_level, certifications,
    job_id, job_title,
    education_score, experience_score, leadership_score,
    certification_score, skills_match_score, industry_relevance_score,
    interview_score, culture_fit, total_score, hired
  FROM {catalog}.{schema}.candidate_scoring_summary
  WHERE candidate_id = get_candidate.candidate_id
""")
print("✅ get_candidate created")

In [ ]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.get_top_candidates(job_id STRING, top_n INT)
RETURNS TABLE (
  rank          INT,
  candidate_id  STRING,
  first_name    STRING,
  last_name     STRING,
  job_id        STRING,
  job_title     STRING,
  total_score   DOUBLE,
  hired         INT
)
COMMENT 'Returns the top N candidates for a given job role ranked by total_score descending. Pass job_id e.g. JR001 and top_n e.g. 5.'
RETURN
  SELECT rank, candidate_id, first_name, last_name, job_id, job_title, total_score, hired
  FROM (
    SELECT
      CAST(ROW_NUMBER() OVER (ORDER BY total_score DESC NULLS LAST) AS INT) AS rank,
      candidate_id, first_name, last_name, job_id, job_title, total_score, hired
    FROM {catalog}.{schema}.candidate_scoring_summary
    WHERE job_id = get_top_candidates.job_id
      AND total_score IS NOT NULL
  )
  WHERE rank <= get_top_candidates.top_n
""")
print("✅ get_top_candidates created")

In [ ]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.get_candidates_by_job(job_id STRING)
RETURNS TABLE (
  candidate_id  STRING,
  first_name    STRING,
  last_name     STRING,
  job_id        STRING,
  job_title     STRING,
  total_score   DOUBLE,
  hired         INT,
  education_level   STRING,
  certifications    STRING,
  years_total_experience INT
)
COMMENT 'Returns all candidates assigned to a specific job role. Pass job_id e.g. JR002.'
RETURN
  SELECT
    candidate_id, first_name, last_name, job_id, job_title,
    total_score, hired, education_level, certifications, years_total_experience
  FROM {catalog}.{schema}.candidate_scoring_summary
  WHERE job_id = get_candidates_by_job.job_id
  ORDER BY total_score DESC NULLS LAST
""")
print("✅ get_candidates_by_job created")

In [ ]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.get_pipeline_candidates()
RETURNS TABLE (
  candidate_id  STRING,
  first_name    STRING,
  last_name     STRING,
  job_id        STRING,
  job_title     STRING,
  education_score          INT,
  experience_score         INT,
  leadership_score         INT,
  certification_score      INT,
  skills_match_score       INT,
  industry_relevance_score INT,
  interview_score          INT,
  culture_fit              INT,
  total_score              DOUBLE
)
COMMENT 'Returns all active pipeline candidates — those with hired IS NULL (C011–C020). These are candidates currently in the hiring process awaiting a decision.'
RETURN
  SELECT
    candidate_id, first_name, last_name, job_id, job_title,
    education_score, experience_score, leadership_score,
    certification_score, skills_match_score, industry_relevance_score,
    interview_score, culture_fit, total_score
  FROM {catalog}.{schema}.candidate_scoring_summary
  WHERE hired IS NULL
  ORDER BY job_id, total_score DESC NULLS LAST
""")
print("✅ get_pipeline_candidates created")

In [ ]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.get_hire_analytics()
RETURNS TABLE (
  job_id                    STRING,
  job_title                 STRING,
  total_candidates          BIGINT,
  hired_count               BIGINT,
  hire_rate_pct             DOUBLE,
  avg_total_score           DOUBLE,
  avg_interview_score       DOUBLE,
  avg_leadership_score      DOUBLE
)
COMMENT 'Returns aggregate hiring statistics by job role: hire rate, average scores, and candidate counts. Only includes historical candidates (hired IS NOT NULL).'
RETURN
  SELECT
    job_id,
    MAX(job_title)                             AS job_title,
    COUNT(*)                                   AS total_candidates,
    SUM(hired)                                 AS hired_count,
    ROUND(AVG(hired) * 100, 1)                 AS hire_rate_pct,
    ROUND(AVG(total_score), 1)                 AS avg_total_score,
    ROUND(AVG(interview_score), 1)             AS avg_interview_score,
    ROUND(AVG(leadership_score), 1)            AS avg_leadership_score
  FROM {catalog}.{schema}.candidate_scoring_summary
  WHERE hired IS NOT NULL
  GROUP BY job_id
  ORDER BY job_id
""")
print("✅ get_hire_analytics created")

## Grant Permissions

The agent's runtime System Service Principal needs:
- `EXECUTE` on each function
- `SELECT` on the underlying Gold table (SQL functions run as INVOKER)

Granting to `account users` covers all identities in the Databricks account, including service principals.

In [ ]:
functions = [
    "get_candidate",
    "get_top_candidates",
    "get_candidates_by_job",
    "get_pipeline_candidates",
    "get_hire_analytics",
]

for fn in functions:
    spark.sql(f"GRANT EXECUTE ON FUNCTION {catalog}.{schema}.{fn} TO `account users`")
    print(f"✅ GRANT EXECUTE on {fn}")

In [ ]:
tables = ["candidate_scoring_summary", "candidates", "job_requirements"]

for tbl in tables:
    spark.sql(f"GRANT SELECT ON TABLE {catalog}.{schema}.{tbl} TO `account users`")
    print(f"✅ GRANT SELECT on {tbl}")

## Smoke Test

In [ ]:
print("--- get_candidate(C001) ---")
display(spark.sql(f"SELECT * FROM {catalog}.{schema}.get_candidate('C001')"))

In [ ]:
print("--- get_top_candidates(JR001, 3) ---")
display(spark.sql(f"SELECT * FROM {catalog}.{schema}.get_top_candidates('JR001', 3)"))

In [ ]:
print("--- get_pipeline_candidates() ---")
display(spark.sql(f"SELECT * FROM {catalog}.{schema}.get_pipeline_candidates()"))

In [ ]:
print("--- get_hire_analytics() ---")
display(spark.sql(f"SELECT * FROM {catalog}.{schema}.get_hire_analytics()"))

## Summary

| Function | Callable as |
|---|---|
| `get_candidate` | `SELECT * FROM bx4.hrd_2030.get_candidate('C005')` |
| `get_top_candidates` | `SELECT * FROM bx4.hrd_2030.get_top_candidates('JR002', 5)` |
| `get_candidates_by_job` | `SELECT * FROM bx4.hrd_2030.get_candidates_by_job('JR003')` |
| `get_pipeline_candidates` | `SELECT * FROM bx4.hrd_2030.get_pipeline_candidates()` |
| `get_hire_analytics` | `SELECT * FROM bx4.hrd_2030.get_hire_analytics()` |

Permissions granted to `account users` — all identities (including the agent's runtime SP) can EXECUTE the functions and SELECT the underlying tables.

**Proceed to** `04_create_genie_space.ipynb`.